In [ ]:
import os
import numpy as np

train_dir = "../data/processed/phase2/train"   # change this

x_file = os.path.join(train_dir, "chb02_01_X.npy")
y_file = os.path.join(train_dir, "chb02_01_y.npy")

X_sample_file = np.load(x_file)
y_sample_file = np.load(y_file)

print("X shape:", X_sample_file.shape)
print("y shape:", y_sample_file.shape)
print("Unique labels:", np.unique(y_sample_file, return_counts=True))

X shape: (1798, 18, 1280)
y shape: (1798,)
Unique labels: (array([0]), array([1798]))


In [8]:
from scipy.stats import skew, kurtosis
from scipy.signal import welch
from scipy.integrate import trapezoid


def extract_bandpower(freqs, psd, fmin, fmax):
    mask = (freqs >= fmin) & (freqs <= fmax)
    if not np.any(mask):
        return 0.0
    return trapezoid(psd[mask], freqs[mask])
def extract_features_from_window(window, sfreq=256):
    """
    window shape: (18, 1280)
    returns: (180,)
    """
    all_features = []

    for ch_signal in window:
        ch_mean = np.mean(ch_signal)
        ch_std = np.std(ch_signal)
        ch_min = np.min(ch_signal)
        ch_max = np.max(ch_signal)
        ch_skew = skew(ch_signal)
        ch_kurt = kurtosis(ch_signal)

        freqs, psd = welch(ch_signal, fs=sfreq, nperseg=min(256, len(ch_signal)))

        delta_power = extract_bandpower(freqs, psd, 0.5, 4)
        theta_power = extract_bandpower(freqs, psd, 4, 8)
        alpha_power = extract_bandpower(freqs, psd, 8, 13)
        beta_power  = extract_bandpower(freqs, psd, 13, 30)

        all_features.extend([
            ch_mean, ch_std, ch_min, ch_max, ch_skew, ch_kurt,
            delta_power, theta_power, alpha_power, beta_power
        ])

    return np.array(all_features, dtype=np.float32)

In [9]:
sample_features = extract_features_from_window(X_sample_file[0], sfreq=256)

print("Feature vector shape:", sample_features.shape)
print("First 10 features:", sample_features[:10])

Feature vector shape: (180,)
First 10 features: [-0.04276878  0.27229634 -0.71232504  0.86599594  0.5442227   0.48731422
  0.02977803  0.01650944  0.00489262  0.00377728]


In [10]:
sample_feature_matrix = np.array([
    extract_features_from_window(window, sfreq=256)
    for window in X_sample_file[:5]
])

print("Feature matrix shape:", sample_feature_matrix.shape)
print("Any NaNs?", np.isnan(sample_feature_matrix).any())
print("Any infs?", np.isinf(sample_feature_matrix).any())

Feature matrix shape: (5, 180)
Any NaNs? False
Any infs? False


step 2

In [17]:
import os
import numpy as np

train_dir = "../data/processed/phase2/train"

all_files = sorted(os.listdir(train_dir))
x_files = [f for f in all_files if f.endswith("_X.npy")]

print("Total train files:", len(x_files))
print("Sample files:", x_files[:5])

Total train files: 175
Sample files: ['chb02_01_X.npy', 'chb02_02_X.npy', 'chb02_03_X.npy', 'chb02_04_X.npy', 'chb02_05_X.npy']


In [18]:
MAX_SAMPLES = 15000

X_features = []
y_labels = []
total_count = 0

for x_file in x_files:
    y_file = x_file.replace("_X.npy", "_y.npy")

    X = np.load(os.path.join(train_dir, x_file))
    y = np.load(os.path.join(train_dir, y_file))

    print(f"Processing {x_file} | X shape: {X.shape} | y shape: {y.shape}")

    for i in range(len(X)):
        features = extract_features_from_window(X[i], sfreq=256)

        X_features.append(features)
        y_labels.append(y[i])

        total_count += 1

        if total_count >= MAX_SAMPLES:
            break

    print(f"Total samples so far: {total_count}")

    if total_count >= MAX_SAMPLES:
        break

X_features = np.array(X_features)
y_labels = np.array(y_labels)

print("Final feature shape:", X_features.shape)
print("Final label shape:", y_labels.shape)
print("Unique labels:", np.unique(y_labels, return_counts=True))
print("Any NaNs?", np.isnan(X_features).any())
print("Any infs?", np.isinf(X_features).any())


Processing chb02_01_X.npy | X shape: (1798, 18, 1280) | y shape: (1798,)
Total samples so far: 1798
Processing chb02_02_X.npy | X shape: (1798, 18, 1280) | y shape: (1798,)
Total samples so far: 3596
Processing chb02_03_X.npy | X shape: (1798, 18, 1280) | y shape: (1798,)
Total samples so far: 5394
Processing chb02_04_X.npy | X shape: (1798, 18, 1280) | y shape: (1798,)
Total samples so far: 7192
Processing chb02_05_X.npy | X shape: (1798, 18, 1280) | y shape: (1798,)
Total samples so far: 8990
Processing chb02_06_X.npy | X shape: (1798, 18, 1280) | y shape: (1798,)
Total samples so far: 10788
Processing chb02_07_X.npy | X shape: (1798, 18, 1280) | y shape: (1798,)
Total samples so far: 12586
Processing chb02_08_X.npy | X shape: (1798, 18, 1280) | y shape: (1798,)
Total samples so far: 14384
Processing chb02_09_X.npy | X shape: (1798, 18, 1280) | y shape: (1798,)
Total samples so far: 15000
Final feature shape: (15000, 180)
Final label shape: (15000,)
Unique labels: (array([0]), array(

In [19]:
unique, counts = np.unique(y_labels, return_counts=True)

print("Class distribution:")
for u, c in zip(unique, counts):
    print(f"Class {u}: {c}")

Class distribution:
Class 0: 15000


step 2 wrong until here - starting again 

In [20]:
MAX_POS = 3000   # seizure windows
MAX_NEG = 12000  # non-seizure windows

X_features = []
y_labels = []

pos_count = 0
neg_count = 0

In [21]:
for x_file in x_files:
    y_file = x_file.replace("_X.npy", "_y.npy")

    X = np.load(os.path.join(train_dir, x_file))
    y = np.load(os.path.join(train_dir, y_file))

    print(f"Processing {x_file}")

    for i in range(len(X)):
        label = y[i]

        if label == 1 and pos_count < MAX_POS:
            features = extract_features_from_window(X[i], sfreq=256)
            X_features.append(features)
            y_labels.append(label)
            pos_count += 1

        elif label == 0 and neg_count < MAX_NEG:
            features = extract_features_from_window(X[i], sfreq=256)
            X_features.append(features)
            y_labels.append(label)
            neg_count += 1

        # Stop early if both reached
        if pos_count >= MAX_POS and neg_count >= MAX_NEG:
            break

    print(f"Pos: {pos_count}, Neg: {neg_count}")

    if pos_count >= MAX_POS and neg_count >= MAX_NEG:
        break

Processing chb02_01_X.npy
Pos: 0, Neg: 1798
Processing chb02_02_X.npy
Pos: 0, Neg: 3596
Processing chb02_03_X.npy
Pos: 0, Neg: 5394
Processing chb02_04_X.npy
Pos: 0, Neg: 7192
Processing chb02_05_X.npy
Pos: 0, Neg: 8990
Processing chb02_06_X.npy
Pos: 0, Neg: 10788
Processing chb02_07_X.npy
Pos: 0, Neg: 12000
Processing chb02_08_X.npy
Pos: 0, Neg: 12000
Processing chb02_09_X.npy
Pos: 0, Neg: 12000
Processing chb02_10_X.npy
Pos: 0, Neg: 12000
Processing chb02_11_X.npy
Pos: 0, Neg: 12000
Processing chb02_12_X.npy
Pos: 0, Neg: 12000
Processing chb02_13_X.npy
Pos: 0, Neg: 12000
Processing chb02_14_X.npy
Pos: 0, Neg: 12000
Processing chb02_15_X.npy
Pos: 0, Neg: 12000
Processing chb02_16+_X.npy
Pos: 43, Neg: 12000
Processing chb02_16_X.npy
Pos: 86, Neg: 12000
Processing chb02_17_X.npy
Pos: 86, Neg: 12000
Processing chb02_18_X.npy
Pos: 86, Neg: 12000
Processing chb02_19_X.npy
Pos: 92, Neg: 12000
Processing chb02_20_X.npy
Pos: 92, Neg: 12000
Processing chb02_21_X.npy
Pos: 92, Neg: 12000
Process

In [22]:
X_features = np.array(X_features)
y_labels = np.array(y_labels)

print("Final shape:", X_features.shape)

unique, counts = np.unique(y_labels, return_counts=True)
print("Class distribution:")
for u, c in zip(unique, counts):
    print(f"Class {u}: {c}")

Final shape: (14104, 180)
Class distribution:
Class 0: 12000
Class 1: 2104


Step-3 Random Forest classifier

In [23]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_features, y_labels)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [25]:
import os
import numpy as np

val_dir = "../data/processed/phase2/val"
all_val_files = sorted(os.listdir(val_dir))
val_x_files = [f for f in all_val_files if f.endswith("_X.npy")]

X_val_features = []
y_val_labels = []

MAX_VAL_SAMPLES = 5000
val_count = 0

for x_file in val_x_files:
    y_file = x_file.replace("_X.npy", "_y.npy")

    X = np.load(os.path.join(val_dir, x_file))
    y = np.load(os.path.join(val_dir, y_file))

    print(f"Processing {x_file}")

    for i in range(len(X)):
        features = extract_features_from_window(X[i], sfreq=256)

        X_val_features.append(features)
        y_val_labels.append(y[i])

        val_count += 1
        if val_count >= MAX_VAL_SAMPLES:
            break

    if val_count >= MAX_VAL_SAMPLES:
        break

X_val_features = np.array(X_val_features)
y_val_labels = np.array(y_val_labels)

print("Validation feature shape:", X_val_features.shape)
print("Validation label shape:", y_val_labels.shape)
print("Validation class distribution:", np.unique(y_val_labels, return_counts=True))
print("Any NaNs?", np.isnan(X_val_features).any())
print("Any infs?", np.isinf(X_val_features).any())

Processing chb12_06_X.npy
Processing chb12_08_X.npy
Processing chb12_09_X.npy
Validation feature shape: (5000, 180)
Validation label shape: (5000,)
Validation class distribution: (array([0, 1]), array([4901,   99]))
Any NaNs? False
Any infs? False


In [26]:
y_val_pred = rf_model.predict(X_val_features)
y_val_proba = rf_model.predict_proba(X_val_features)[:, 1]

In [27]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("Confusion Matrix:")
print(confusion_matrix(y_val_labels, y_val_pred))

print("\nClassification Report:")
print(classification_report(y_val_labels, y_val_pred, digits=4))

if len(np.unique(y_val_labels)) == 2:
    print("\nROC-AUC:", roc_auc_score(y_val_labels, y_val_proba))
else:
    print("\nROC-AUC cannot be computed because validation has only one class.")

Confusion Matrix:
[[4200  701]
 [  62   37]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9855    0.8570    0.9167      4901
           1     0.0501    0.3737    0.0884        99

    accuracy                         0.8474      5000
   macro avg     0.5178    0.6154    0.5026      5000
weighted avg     0.9669    0.8474    0.9003      5000


ROC-AUC: 0.6141109111931393


In [28]:
import pandas as pd

importances = rf_model.feature_importances_

feature_df = pd.DataFrame({
    "feature_index": np.arange(len(importances)),
    "importance": importances
}).sort_values("importance", ascending=False)

print(feature_df.head(20))

     feature_index  importance
66              66    0.044674
27              27    0.039437
77              77    0.036442
171            171    0.034848
37              37    0.034624
106            106    0.033175
161            161    0.031771
176            176    0.031486
78              78    0.028898
139            139    0.028247
149            149    0.027646
169            169    0.025112
166            166    0.024836
141            141    0.024495
101            101    0.024464
107            107    0.020963
99              99    0.019659
142            142    0.018833
38              38    0.017601
61              61    0.017581


Changing some parameters for random forest classifier

In [37]:
rf_model_2 = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf_model_2.fit(X_features, y_labels)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [34]:
import os
import numpy as np

val_dir = "../data/processed/phase2/val"
all_val_files = sorted(os.listdir(val_dir))
val_x_files = [f for f in all_val_files if f.endswith("_X.npy")]

X_val_features = []
y_val_labels = []

MAX_VAL_SAMPLES = 5000
val_count = 0

for x_file in val_x_files:
    y_file = x_file.replace("_X.npy", "_y.npy")

    X = np.load(os.path.join(val_dir, x_file))
    y = np.load(os.path.join(val_dir, y_file))

    print(f"Processing {x_file}")

    for i in range(len(X)):
        features = extract_features_from_window(X[i], sfreq=256)

        X_val_features.append(features)
        y_val_labels.append(y[i])

        val_count += 1
        if val_count >= MAX_VAL_SAMPLES:
            break

    if val_count >= MAX_VAL_SAMPLES:
        break

X_val_features = np.array(X_val_features)
y_val_labels = np.array(y_val_labels)

print("Validation feature shape:", X_val_features.shape)
print("Validation label shape:", y_val_labels.shape)
print("Validation class distribution:", np.unique(y_val_labels, return_counts=True))
print("Any NaNs?", np.isnan(X_val_features).any())
print("Any infs?", np.isinf(X_val_features).any())

Processing chb12_06_X.npy
Processing chb12_08_X.npy
Processing chb12_09_X.npy
Validation feature shape: (5000, 180)
Validation label shape: (5000,)
Validation class distribution: (array([0, 1]), array([4901,   99]))
Any NaNs? False
Any infs? False


In [38]:
y_val_pred = rf_model_2.predict(X_val_features)
y_val_proba = rf_model_2.predict_proba(X_val_features)[:, 1]

In [39]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("Confusion Matrix:")
print(confusion_matrix(y_val_labels, y_val_pred))

print("\nClassification Report:")
print(classification_report(y_val_labels, y_val_pred, digits=4))

if len(np.unique(y_val_labels)) == 2:
    print("\nROC-AUC:", roc_auc_score(y_val_labels, y_val_proba))
else:
    print("\nROC-AUC cannot be computed because validation has only one class.")

Confusion Matrix:
[[4145  756]
 [  61   38]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9855    0.8457    0.9103      4901
           1     0.0479    0.3838    0.0851        99

    accuracy                         0.8366      5000
   macro avg     0.5167    0.6148    0.4977      5000
weighted avg     0.9669    0.8366    0.8940      5000


ROC-AUC: 0.6442140235243683


Recall 

Precision = $\frac{TP}{TP + FP}$

text{Recall} = $\frac{TP}{TP + FN}$

This is important as we are looking into how many positive seizure windows are correctly identified and how many positive windows are not identified. (as missing seizure is not acceptable). This is the reason we are not looking into precetion as it only deals with how many positives are computed correctly and does not count false negetive.

Here random forest was not able to do properly because, they are made to work better on tabular data not sequential or time series data. so we can its not undestanding the time series data like continuous windows, frequency featurs, channel differentiations etc... 

	Dataset is heavily imbalanced → very few seizure windows compared to normal
	•	Accuracy is misleading → model can predict all non-seizure and still look “good”
	•	Recall = how many actual seizures we correctly detect
	•	In this problem, missing a seizure (false negative) is much worse than a false alarm
	•	So recall is the primary metric, not accuracy or precision
	•	Our recall ≈ 0.38–0.40
	•	Meaning: model detects only ~40% of actual seizures
	•	Misses ~60% → not acceptable for real-world use
	•	Why it fails:
	1.	We compressed raw EEG (18 × 1280) into 180 features → lost temporal patterns
	2.	Features only capture summary stats and band power → miss waveform structure
	3.	Random Forest treats data as flat → no understanding of sequence or signal evolution
	•	Result:
→ model learns some signal (ROC-AUC ~0.6)
→ but cannot capture full seizure dynamics
	•	Conclusion:
→ feature-based ML has a ceiling
→ need models that learn from raw time-series (CNN/LSTM)